# Seeing Things: UFO Reports as Human Reporting Data

This notebook is the technical explainer behind the public website. The website tells the story for a non-technical audience; this notebook documents the data source, cleaning choices, analysis logic, narrative genre, limitations, and references.

## 1. Motivation

The project treats UFO sightings as social data. A UFO report is not only a claim about something in the sky; it is also the result of a person observing, interpreting, remembering, and submitting a report through a particular reporting system.

**Research question:** Are UFO reports randomly scattered, or are they structured by time, place, shape language, duration, and reporting infrastructure?

**Audience:** curious non-technical readers who should be able to understand the findings from the website alone.

**Goal:** use narrative visualization to show that the dataset is patterned, biased, and deeply human without claiming that the reports verify UFO events. The updated story focuses especially on why the United States dominates the archive and what changes before and after internet-era reporting.


## 2. Basic Stats

Maven Analytics describes the UFO Sightings dataset as 80,000+ records between 1949 and 2014, with city, state, country, coordinates, shape, duration, date/time, and comments. The raw file used in this project is the archived NUFORC scrubbed CSV from Zenodo, which follows the same data lineage. The raw file has no header, so the processing script assigns the classic NUFORC scrubbed column names.

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
from collections import Counter

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw' / 'ufo_sightings_raw.csv'
CLEAN = ROOT / 'data' / 'processed' / 'ufo_sightings_clean.csv'
SUMMARY = ROOT / 'site' / 'data' / 'summary.json'

subprocess.run([sys.executable, str(ROOT / 'scripts' / 'build_data.py')], check=True)
summary = json.loads(SUMMARY.read_text())
summary


### Cleaning Choices

- Parse the raw `datetime` field, including the NUFORC convention where some rows use hour `24`, which is rolled to `00:00` on the following day.
- Extract `year`, `month`, `weekday`, and `hour` for temporal analysis.
- Standardize country codes, state codes, and shape labels.
- Convert duration seconds to numeric values, marking invalid or negative durations as missing.
- Keep comments as text after HTML entity cleanup.
- Exclude invalid coordinates from map aggregates, but document how many were removed.
- Preserve the raw CSV unchanged in `data/raw/` and write a cleaned analysis file to `data/processed/`.

In [ ]:
with CLEAN.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"Raw rows: {summary['raw_rows']:,}")
print(f"Clean rows: {summary['clean_rows']:,}")
print(f"Valid coordinate rows: {summary['valid_geo_rows']:,}")
print(f"Actual date range: {summary['date_range']['min_year']}-{summary['date_range']['max_year']}")
print(f"Maven stated range: {summary['maven_stated_range']}")
print(f"Documented cleaning issues: {summary['issues']}")

The actual raw file contains reports from 1906 to 2014, while Maven states 1949 to 2014. This mismatch is treated as a provenance and data-quality issue, not hidden. It supports the broader point that the data is a historical reporting record rather than a perfectly curated event database.

## 3. Data Analysis

The analysis focuses on seven linked questions: when reports spike, what people say they saw, whether reports follow human rhythms, why the United States dominates, which US states and cities stand out, how duration changes by shape and era, and whether hotspots persist over time.


In [ ]:
def load_site_json(name):
    return json.loads((ROOT / 'site' / 'data' / name).read_text())

annual = load_site_json('annual_reports.json')
shapes = load_site_json('shape_counts.json')
countries = load_site_json('country_counts.json')
states = load_site_json('state_counts.json')
cities = load_site_json('city_counts.json')
duration_shapes = load_site_json('duration_by_shape.json')
duration_eras = load_site_json('duration_by_era_shape.json')
area51 = load_site_json('area51_summary.json')
hotspots = load_site_json('hotspots.json')

peak_year = max(annual, key=lambda d: d['reports'])
top_shapes = shapes[:8]
top_countries = countries[:5]
top_persistent = hotspots[:5]

print('Peak reporting year:', peak_year)
print('Top shapes:', top_shapes)
print('Top countries:', top_countries)
print('Top US cities:', cities[:8])
print('Duration by shape:', duration_shapes[:6])
print('Era comparison:', duration_eras)
print('Area 51 summary:', area51)
print('Most persistent map bins:', top_persistent)


### Exploratory Plots

The figures below are generated from the same compact JSON files used by the website. They make the notebook readable on its own and show the main empirical patterns behind the narrative.

In [ ]:
# Rebuild the notebook SVG figures from the website-ready data.
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'build_notebook_figures.py')], check=True)


#### Reports by year

The yearly trend shows why the project treats the data as a reporting system. Reports increase sharply in the 1990s and 2000s, which is consistent with easier online reporting and growing visibility of UFO reporting systems, but not proof that the sky itself changed.

The project uses 1995 as a practical internet-era split. Before 1995 there are far fewer reports and the median reported duration is longer; from 1995 onward, the archive becomes much denser and short observations appear more often.

![Reports by year](figures/annual_reports.svg)


#### Top reported shapes

Shape labels reveal the vocabulary people use to classify uncertainty. `light` dominates overall, reinforcing that many reports describe distant or ambiguous visual phenomena. The shape vocabulary also changes over time: before 1995, `disk` is the leading shape; from 1995 to 2014, `light` becomes dominant.

![Top reported shapes](figures/shape_counts.svg)


#### Month-hour rhythm

The heatmap connects reports to human routines: sightings cluster around evening and seasonal windows when people are more likely to be outside and looking upward.

![Reports by month and hour](figures/month_hour_heatmap.svg)

#### Hotspot persistence

The persistence view separates high-volume bins from bins that keep appearing across many different years. This distinction addresses whether reports recur in the same places over time. It also keeps the global map as perspective while the main geographic interpretation now focuses on the United States.

![Hotspot persistence](figures/hotspot_persistence.svg)


#### USA state and city geography

The United States accounts for about 81% of clean reports, so the updated website zooms into US geography instead of leaving the map at a broad world overview. Raw state counts are led by large states such as California, Florida, Texas, and New York, but reports per million people bring Washington, Oregon, Montana, Alaska, Maine, Vermont, Arizona, New Mexico, and other smaller states forward.

City-level hotspots make the geography more concrete. Seattle, Phoenix, Las Vegas, Los Angeles, San Diego, Portland, Houston, and Chicago are among the largest city clusters.

![US city hotspots](figures/city_hotspots.svg)


#### Area 51 as a reality check

Area 51 is useful as a cultural reference, but it is not the main count story in this dataset. Using a simple bounding region around the Area 51/Rachel/Groom Lake area, there are about 20 nearby reports. Nevada has 803 reports overall, and Las Vegas alone has 363. This makes Area 51 a good sidebar: famous in the imagination, small in the archive.


#### Duration by shape

Reported duration is noisy because it is estimated by people, but it still reveals a relationship between event description and event length. Among common shapes, `changing` reports have the longest median duration, followed by shapes such as `diamond` and `disk`. Short labels such as `flash` and `chevron` tend to have short medians.

![Duration by shape](figures/duration_by_shape.svg)


#### Before and after internet-era reporting

Using 1995 as the split, the archive changes strongly. Pre-internet reports have a median duration around 300 seconds and `disk` is the leading shape. In the 1995-2014 internet era, median duration drops to about 180 seconds and `light` becomes the leading shape. This is interpreted as a change in reporting behavior and vocabulary, not proof that physical events changed.

![Era comparison](figures/era_comparison.svg)


### Bias Analysis

| Bias | Why it matters | How the project handles it |
| --- | --- | --- |
| US reporting-system bias | About 81% of clean reports are coded as United States. | Treat US dominance as evidence of NUFORC visibility, English-language access, population, internet access, and reporting culture. |
| Population bias | More people means more possible reports. | Show raw geography carefully and include a simple US population-normalized state view. |
| Internet/reporting bias | Online reporting changed who could submit reports and when. | Compare pre-1995 with 1995-2014 and avoid causal certainty. |
| Language bias | English-speaking countries are overrepresented. | Discuss country concentration as reporting-system coverage, not true UFO activity. |
| Geocoding bias | Coordinates may point to city centers rather than exact locations. | Use spatial bins and city/state aggregates instead of exact point claims. |
| Selection bias | Only people who report are included. | Describe records as reports, never verified sightings. |
| Visibility bias | Darkness, weather, season, and outdoor activity affect observation. | Use month-hour patterns as behavioral context. |


### Hotspot Persistence

A simple count can confuse a one-time burst with a place that keeps appearing. The website therefore uses spatial bins and computes:

- `total_reports`
- `active_years`
- `active_decades`
- `persistence_ratio = active_years / observed_year_span`
- `burstiness`, the share of a bin's reports concentrated in its busiest year

This is enough to answer the assignment feedback about whether sightings recur in the same places over time without overclaiming a predictive model.

## 4. Genre

The website uses an **interactive slideshow / martini glass** structure, following Segel and Heer. The opening chapters are author-guided: the reader moves through a clear argument about report data, time, language, human rhythms, USA-focused geography, duration, and internet-era reporting. The final hotspot chapter opens into exploration with a map, time slider, and mode toggle.

Narrative visualization techniques used:

- **Visual structuring:** sections are ordered as chapters with one main question each.
- **Highlighting:** annotations call attention to the internet-era rise, the Maven date-range mismatch, USA dominance, and the Area 51 contrast.
- **Transition guidance:** the story moves from raw spectacle to skepticism, then to structured evidence.
- **Interactivity:** the reader can toggle normalized geography and explore hotspot persistence.
- **Messaging:** every visualization has a non-technical takeaway and caveat.


## 5. Visualizations

| Visualization | Purpose | Interaction | Limitation |
| --- | --- | --- | --- |
| Hero map | Create the initial sense of geographic pattern. | Static overview. | Raw geography can be mistaken for true activity. |
| Dataset metrics | Ground the reader in row count, date range, US share, and top shape. | Static. | Summary stats cannot show every cleaning choice. |
| Annual line chart | Show temporal growth and reporting spikes. | Static annotation. | Cannot prove the internet caused the rise. |
| Shape bar chart | Show the vocabulary of reported sightings. | Static. | Shape labels are self-reported and messy. |
| Month-hour heatmap | Show human observation rhythms. | Static. | Does not include weather or actual sky-watching rates. |
| State ranking | Compare raw counts with reports per million. | Toggle raw/per-million. | 2010 population is an imperfect historical denominator. |
| City hotspots | Move the US map from broad states toward recognizable places. | Static ranking. | City counts inherit geocoding and population bias. |
| Duration by shape | Compare median duration across common shape labels. | Static ranking. | Durations are human estimates and can contain extreme values. |
| Era comparison | Compare pre-1995 reports with 1995-2014 reports. | Static summary cards. | The internet-era split is interpretive, not a causal proof. |
| Hotspot map | Distinguish total reports from persistent places. | Time slider, count/persistence toggle, hover tooltips. | Approximate hex bins are not equal-area H3 cells. |
| Country ranking | Provide a final global perspective. | Static ranking. | Country coverage reflects the reporting pipeline. |


## 6. Discussion

What went well: the dataset supports a strong social-data story because temporal, geographic, linguistic, duration, and reporting-system patterns all point in the same direction. The United States focus makes the geographic argument clearer: raw counts identify populous and reporting-heavy states, normalized counts reveal smaller high-rate states, and city hotspots show recognizable urban reporting clusters.

Main findings: about 81% of clean reports are coded as US reports; California leads raw state counts, while Washington and several smaller western/northern states stand out per million people. Seattle, Phoenix, Las Vegas, Los Angeles, San Diego, Portland, Houston, and Chicago are among the largest city hotspots. Area 51 is culturally famous, but the nearby bounding region has only about 20 reports in this data, compared with 363 reports from Las Vegas. Duration also changes: among common shapes, `changing` has the longest median duration, while `flash` and `chevron` are short; pre-1995 reports have a median duration of about 300 seconds, while the 1995-2014 internet era has a median around 180 seconds.

What is missing: stronger causal claims would require external datasets such as historical internet adoption, media coverage, local population by year, weather, light pollution, or aircraft/airport activity. The current population normalization is deliberately simple and should be presented as a caveat rather than a final correction.

Main limitation: the records are reports, not verified UFO events. The website therefore avoids saying where UFOs actually are and instead asks what the reports reveal about people, places, and reporting behavior. The playful closing joke is aimed at American reporting culture: Americans may not be uniquely good at spotting aliens, but they are very good at seeing something weird and finding a form to submit.


## 7. Contributions

Replace these placeholders with the final group member names before submission:

- Person A: data cleaning, provenance audit, notebook structure.
- Person B: D3 charts, hotspot aggregation, map/time-slider interaction.
- Person C: website narrative, design polish, references, final writing.
- Together: interpretation, limitation review, presentation rehearsal.

## 8. References

- Course final project requirements: https://github.com/suneman/socialdata2026/wiki/Final-Project
- Maven UFO Sightings dataset page: https://mavenanalytics.io/data-playground/ufo-sightings
- Archived NUFORC scrubbed CSV on Zenodo: https://zenodo.org/records/1205624
- Segel, E. and Heer, J. Narrative Visualization: Telling Stories with Data: https://idl.uw.edu/papers/narrative
- 2010 US Census state populations, used as a simple state denominator.